# This document focuses on analyzing and finding the answers to the Multi choice questions in Part 1 of the assignment

## Prepare libraries and load data:

In [511]:
import pandas as pd

df_cus = pd.read_csv('../dataset1/customers.csv')
df_geo = pd.read_csv('../dataset1/geography.csv')
df_item = pd.read_csv('../dataset1/order_items.csv')
df_ord = pd.read_csv('../dataset1/orders.csv')
df_pay = pd.read_csv('../dataset1/payments.csv')
df_pduct = pd.read_csv('../dataset1/products.csv')
df_pmot = pd.read_csv('../dataset1/promotions.csv')
df_ret = pd.read_csv('../dataset1/returns.csv')
df_ship = pd.read_csv('../dataset1/shipments.csv')
df_web = pd.read_csv('../dataset1/web_traffic.csv')


C:\Users\thaim\AppData\Local\Temp\ipykernel_8256\2399593915.py:5: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_item = pd.read_csv('../dataset1/order_items.csv')


Because the information about the data has already been described in the assignment, we will limit checking the data's `info()`

## Question 1: 
Among customers with more than one order, what is the approximate median number of days between two consecutive purchases (inter-order gap)? (Calculated from `orders.csv`)

In [512]:
# Extract necessary columns
df_ques1 = df_ord[['order_id', 'customer_id', 'order_date']].copy()
# Inspect data types and missing values
df_ques1.info()
display(df_ques1.head(5))

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   order_id     646945 non-null  int64
 1   customer_id  646945 non-null  int64
 2   order_date   646945 non-null  str  
dtypes: int64(2), str(1)
memory usage: 14.8 MB


,order_id,customer_id,order_date
0,1,58578,2012-07-04
1,2,58621,2012-07-04
2,3,58811,2012-07-04
3,4,59453,2012-07-04
4,6,57821,2012-07-06


In [513]:
# Convert data types for calculation
df_ques1['order_date'] = pd.to_datetime(df_ques1['order_date'], format='%Y-%m-%d')
# Keep only customers with multiple orders and sort them
df_ques1 = df_ques1[df_ques1.duplicated(subset=['customer_id'], keep=False)].sort_values(by=['customer_id', 'order_date', 'order_id'])
# Check order frequency per customer
print(df_ques1['customer_id'].value_counts())

customer_id
139050    107
141899    105
141897    103
141898    100
139138     96
         ... 
157502      2
157503      2
157505      2
157530      2
157555      2
Name: count, Length: 67888, dtype: int64


In [514]:
# Calculate the time gap between consecutive purchases for each customer
df_ques1['inter_order_gap'] = df_ques1.groupby('customer_id')['order_date'].diff().dt.days
# Drop the first purchase of each customer (where the gap is NaN)
df_ques1 = df_ques1[df_ques1.groupby('customer_id').cumcount() > 0]
# Check the first 5 rows
display(df_ques1.head(5))

,order_id,customer_id,order_date,inter_order_gap
143252,184922,1,2014-05-31,675.0
238890,308113,1,2015-07-31,426.0
374571,483190,1,2017-04-23,632.0
544446,702081,1,2020-02-24,1037.0
586950,756884,1,2021-04-24,425.0


In [515]:
df_ques1.describe()

,order_id,customer_id,order_date,inter_order_gap
count,556699.000000,556699.000000,556699,556699.000000
mean,450239.692944,86035.872175,2017-04-05 16:38:14.536903,285.592509
min,153.000000,1.000000,2012-07-04 00:00:00,0.000000
25%,259181.500000,41989.000000,2015-03-19 00:00:00,46.000000
50%,456552.000000,91618.000000,2016-12-30 00:00:00,144.000000
75%,647013.500000,134426.000000,2019-01-28 00:00:00,357.000000
max,834397.000000,157563.000000,2022-12-31 00:00:00,3785.000000
std,227928.945563,48735.831389,NaN,389.691558


In the descriptive statistics table, for the `inter_order_gap` column, the median value (50%) = **144** (days)
#### -> Choose C

## Question 2: 
Which product segment in `products.csv` has the highest average gross profit margin, using the formula (`price` - `cogs`)/`price`?

In [516]:
# Extract necessary columns
df_ques2 = df_pduct[['segment', 'price', 'cogs']].copy()
# Inspect data types and missing values
df_ques2.info()
# Check the unique values and their frequencies in the segment column
print(df_ques2['segment'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   segment  2412 non-null   str    
 1   price    2412 non-null   float64
 2   cogs     2412 non-null   float64
dtypes: float64(2), str(1)
memory usage: 56.7 KB
segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148
Name: count, dtype: int64


In [517]:
# Calculate and store the gross profit margin
df_ques2['gross_profit_margin'] = (df_ques2['price'] - df_ques2['cogs']) / df_ques2['price']
# Calculate the average gross profit margin for each segment
mean_by_segment = df_ques2.groupby('segment')['gross_profit_margin'].mean()
# Print result
print(mean_by_segment)
print(f"\nHighest segment: {mean_by_segment.idxmax()} with margin {mean_by_segment.max()}")

segment
Activewear     0.265600
All-weather    0.284176
Balanced       0.258038
Everyday       0.236343
Performance    0.263650
Premium        0.285377
Standard       0.313442
Trendy         0.240758
Name: gross_profit_margin, dtype: float64

Highest segment: Standard with margin 0.31344174843884803


#### -> Choose D

## Question 3: 
Among the return records linked to products in the *Streetwear* category (joining `returns` with `products` on `product_id`), which return reason appears the most?

In [518]:
# Extract necessary columns
df_ques3 = df_ret[['return_id', 'product_id', 'return_reason']].copy()
df_support = df_pduct[['product_id', 'category']].copy()
# Inspect data types and missing values
df_ques3.info()
df_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   return_id      39939 non-null  str  
 1   product_id     39939 non-null  int64
 2   return_reason  39939 non-null  str  
dtypes: int64(1), str(2)
memory usage: 936.2 KB
<class 'pandas.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   product_id  2412 non-null   int64
 1   category    2412 non-null   str  
dtypes: int64(1), str(1)
memory usage: 37.8 KB


In [519]:
# Merge product categories into returns using product_id
df_ques3 = df_ques3.merge(df_support, on='product_id', how='left')
# Filter for Streetwear products
df_ques3 = df_ques3[df_ques3['category'] == 'Streetwear']
# Print result
print(df_ques3['return_reason'].value_counts())
mode_reason = df_ques3['return_reason'].mode()[0]
print("\nThe most common reason for returns is:", mode_reason)

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

The most common reason for returns is: wrong_size


#### -> Choose B

## Question 4: 
In `web_traffic.csv`, which `traffic source` has the lowest average `bounce_rate` across all days that source appears in the `traffic_source` column?

In [520]:
# Extract necessary columns
df_ques4 = df_web[['date','bounce_rate', 'traffic_source']].copy()
# Inspect data types and missing values
df_ques4.info()
display(df_ques4.head(5))

<class 'pandas.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            3652 non-null   str    
 1   bounce_rate     3652 non-null   float64
 2   traffic_source  3652 non-null   str    
dtypes: float64(1), str(2)
memory usage: 85.7 KB


,date,bounce_rate,traffic_source
0,2013-01-01,0.00514,organic_search
1,2013-01-02,0.00406,organic_search
2,2013-01-03,0.00401,direct
3,2013-01-04,0.00562,direct
4,2013-01-05,0.00525,referral


In [521]:
# Convert date column to datetime format
df_ques4['date'] = pd.to_datetime(df_ques4['date'], format='%Y-%m-%d')
# Calculate the average bounce rate per traffic source
mean_by_traffic_source = df_ques4.groupby('traffic_source')['bounce_rate'].mean()
# Print result
print(mean_by_traffic_source)
print(f"\nTraffic source: {mean_by_traffic_source.idxmin()} has the lowest average bounce rate of {mean_by_traffic_source.min()}")

traffic_source
direct            0.004511
email_campaign    0.004458
organic_search    0.004504
paid_search       0.004478
referral          0.004499
social_media      0.004476
Name: bounce_rate, dtype: float64

Traffic source: email_campaign has the lowest average bounce rate of 0.0044584356435643565


#### -> Choose C

## Question 5: 
What is the approximate percentage of rows in `order_items.csv` that have a promotion applied (i.e., `promo_id` is not null)?

In [522]:
df_ques5 = df_item.copy()
# Inspect data types and missing values
df_ques5.info()

<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  str    
 6   promo_id_2       206 non-null     str    
dtypes: float64(2), int64(3), str(2)
memory usage: 38.2 MB


In [523]:
percentage_apply_promotion = df_ques5['promo_id'].notna().sum() / len(df_ques5) * 100
print(f"The percentage of the promotion applied is: {percentage_apply_promotion.round(0)}%")

The percentage of the promotion applied is: 39.0%


#### -> Choose C

## Question 6: 
In `customers.csv`, considering customers with a non-null `age_group`, which age group has the highest average number of orders per customer? (total orders / number of customers in the group)

In [524]:
# Extract necessary columns
df_ques6 = df_ord[['customer_id', 'order_id']].copy()
df_support = df_cus[['customer_id', 'age_group']].copy()
# Inspect data types and missing values
df_ques6.info()
df_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   customer_id  646945 non-null  int64
 1   order_id     646945 non-null  int64
dtypes: int64(2)
memory usage: 9.9 MB
<class 'pandas.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   customer_id  121930 non-null  int64
 1   age_group    121930 non-null  str  
dtypes: int64(1), str(1)
memory usage: 1.9 MB


In [525]:
# Merge customer age group into the orders dataframe using customer_id
df_ques6 = df_ques6.merge(df_support, on='customer_id', how='left')
# Calculate average orders per customer for each age group
mean_by_age_group = df_ques6.groupby('age_group')['order_id'].count() / df_ques6.groupby('age_group')['customer_id'].nunique()
# Print result
print(mean_by_age_group)
print(f"Age group: {mean_by_age_group.idxmax()} has the highest average orders")

age_group
18-24    7.068577
25-34    7.112230
35-44    7.206159
45-54    7.220264
55+      7.268731
dtype: float64
Age group: 55+ has the highest average orders


#### -> Choose A

## Question 7: 
Which `region` in `geography.csv` generates the highest total revenue in `sales_train.csv`?

In [526]:
# Extract necessary columns
df_ques7 = df_ord[['order_id', 'order_date', 'zip', 'order_status']].copy()
df_support = df_pay[['order_id', 'payment_value']].copy()
df_support2 = df_geo[['zip', 'region']].copy()
# Inspect data types and missing values
df_ques7.info()
df_support.info()
df_support2.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   order_id      646945 non-null  int64
 1   order_date    646945 non-null  str  
 2   zip           646945 non-null  int64
 3   order_status  646945 non-null  str  
dtypes: int64(2), str(2)
memory usage: 19.7 MB
<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 2 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       646945 non-null  int64  
 1   payment_value  646945 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 9.9 MB
<class 'pandas.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   zip     39948 non-null  int64
 1   region  39948 non-null  str  
dtypes: int64(1), str(1)
memory usag

In [527]:
# Convert data types for calculation
df_ques7['order_date'] = pd.to_datetime(df_ques7['order_date'], format='%Y-%m-%d')
# Merge payment and geography data into the orders dataframe
df_ques7 = df_ques7.merge(df_support, on='order_id', how='left')
df_ques7 = df_ques7.merge(df_support2, on='zip', how='left')
# Print to check data types and detect missing data
df_ques7.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   order_id       646945 non-null  int64         
 1   order_date     646945 non-null  datetime64[us]
 2   zip            646945 non-null  int64         
 3   order_status   646945 non-null  str           
 4   payment_value  646945 non-null  float64       
 5   region         646945 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), str(2)
memory usage: 29.6 MB


In [528]:
# Calculate total revenue per region within the timeframe (04/07/2012 - 31/12/2022)
revenue = df_ques7.groupby('region')['payment_value'].sum()
# Print result
print(revenue)
print(f"\nRegion: {revenue.idxmax()} has the highest revenue")

region
Central    4.719491e+09
East       7.291151e+09
West       3.670227e+09
Name: payment_value, dtype: float64

Region: East has the highest revenue


#### -> Choose C

## Question 8: 
Among the orders with `order_status` = *cancelled* in `orders.csv`, which payment method is used the most?

In [529]:
# Extract necessary columns
df_ques8 = df_ord[['order_id', 'order_status', 'payment_method']].copy()
# Filter for cancelled orders
df_ques8 = df_ques8[df_ques8['order_status'] == 'cancelled']
# Inspect data types and missing values
df_ques8.info()

<class 'pandas.DataFrame'>
Index: 59462 entries, 16 to 646926
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        59462 non-null  int64
 1   order_status    59462 non-null  str  
 2   payment_method  59462 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.8 MB


In [530]:
# Print result
print(df_ques8['payment_method'].value_counts())
mode_payment = df_ques8['payment_method'].mode()[0]
print("\nThe most commonly used payment method is:", mode_payment)

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

The most commonly used payment method is: credit_card


#### -> Choose A

## Question 9: 
Among the four product sizes (*S*, *M*, *L*, *XL*), which size has the highest return rate, defined as the number of records in `returns` divided by the number of rows in `order_items` (joined with `products` on `product_id`)?

In [531]:
# Extract necessary columns
df_ques9_ret = df_ret[['return_id', 'order_id', 'product_id', 'return_quantity']].copy()
df_ques9_item = df_item[['order_id' ,'product_id', 'quantity']].copy()
df_support = df_pduct[['product_id', 'size']].copy()
# Inspect data types and missing values
df_ques9_ret.info()
df_ques9_item.info()
df_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   return_id        39939 non-null  str  
 1   order_id         39939 non-null  int64
 2   product_id       39939 non-null  int64
 3   return_quantity  39939 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   order_id    714669 non-null  int64
 1   product_id  714669 non-null  int64
 2   quantity    714669 non-null  int64
dtypes: int64(3)
memory usage: 16.4 MB
<class 'pandas.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   product_id  2412 non-null   int64
 1   size        2412 non-null   str  
dtyp

In [532]:
# Merge product size into the returns dataframe using product_id
df_ques9_ret = df_ques9_ret.merge(df_support, on='product_id', how='left')
# Merge product size into the order items dataframe using product_id
df_ques9_item = df_ques9_item.merge(df_support, on='product_id', how='left')
# Count total order items sold per size
sale = df_ques9_item.groupby('size').size()
# Count total returns per size
returns = df_ques9_ret.groupby('size').size()
# Print result
print(returns / sale)
print(f"\nThe product size with the highest return rate is: {(returns / sale).idxmax()}")

size
L     0.056250
M     0.055660
S     0.056515
XL    0.055200
dtype: float64

The product size with the highest return rate is: S


#### -> Choose A

## Question 10: 
In `payments.csv`, which installment plan has the highest average payment value per order?

In [533]:
df_ques10 = df_pay.copy()
# Inspect data types and missing values
df_ques10.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  str    
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 19.7 MB


In [534]:
# Tính toán giá trị thanh toán trung bình trên mỗi đơn hàng trong tất cả các kế hoạch trả góp
average = df_pay.groupby('installments')['payment_value'].mean()
# Print result
print(average)
print(f"\nThe installment plan with the highest average payment is: {average.idxmax()}")

installments
1     24113.274166
2       708.473729
3     24399.635486
6     24446.654403
12    24245.772694
Name: payment_value, dtype: float64

The installment plan with the highest average payment is: 6


#### -> Choose C